# ClusterBP v2: other sequence models instead of the GRU

Why a GRU in the sequence branch? Comparison with three alternatives on the same inputs (last 20 measurements, 5 channels) and the same training scheme:

- **TimeAwareGRU** — GRU with hidden-state decay over the time between measurements (as in GRU-D, Che et al. 2018, and DATA-GRU, Tan et al. 2020: accounting for irregular sampling);
- **TCN** — convolutional network with causal dilated convolutions;
- **Transformer** — a small encoder with attention over all measurements of the window.

Each model is compared with the GRU both alone and stacked with XGB_full (as in the final ClusterBP v2 model): what matters is whether a more complex sequence model changes the quality of the whole ensemble.
Folds, inner validation, negative subsampling and early stopping are as in `clusterbp_v2.ipynb`. GRU and XGB_full are taken from the already trained fold models.

In [1]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

from clusterbp_lib import (RESULTS_DIR, SEED, N_FOLDS, FEATURE_SETS, SEQ_MODELS, load_cached, fold_masks,
                           add_phenotype_columns, load_gru, load_xgb, train_seq_model, gru_logits, sigmoid, logit,
                           best_threshold, boot_eval, ci)

warnings.filterwarnings('ignore')
torch.set_num_threads(os.cpu_count())
SEQ_TARGETS = ['A', 'B']
NEW = ['TimeAwareGRU', 'TCN', 'Transformer']

win, X_seq, seq_len, pheno_win, oof = load_cached()
pid = win['pid'].values

In [2]:
store = {t: {} for t in SEQ_TARGETS}
info = []


def put(t, name, te, s, p):
    st = store[t].setdefault(name, [np.full(len(win), np.nan), np.zeros(len(win), np.int8)])
    st[0][te] = s[te]
    st[1][te] = p[te]


t_all = time.time()
for k in range(N_FOLDS):
    tr, va, te = fold_masks(win, k)
    add_phenotype_columns(win, pheno_win, k)
    for t in SEQ_TARGETS:
        y = win[f'y_{t}'].values
        px = load_xgb('full', t, k).predict_proba(win[FEATURE_SETS['full']].values)[:, 1]
        gm, mu, sd, corr = load_gru(t, k)
        seq_probs = {'GRU': sigmoid(gru_logits(gm, X_seq, seq_len, mu, sd) + corr)}
        for name in NEW:
            t0 = time.time()
            m, mu_, sd_, corr_, log = train_seq_model(SEQ_MODELS[name], X_seq[tr], seq_len[tr], y[tr],
                                                      X_seq[va], seq_len[va], y[va], SEED + k)
            seq_probs[name] = sigmoid(gru_logits(m, X_seq, seq_len, mu_, sd_) + corr_)
            info.append({'target': t, 'fold': k, 'model': name, 'epochs': len(log),
                         'parameters': sum(p.numel() for p in m.parameters()), 'seconds': time.time() - t0})
        for name, ps in seq_probs.items():
            thr, _ = best_threshold(y[va], ps[va])
            put(t, name, te, ps, (ps > thr).astype(np.int8))
            st = LogisticRegression(max_iter=3000).fit(np.column_stack([logit(ps[va]), logit(px[va])]), y[va])
            s = st.predict_proba(np.column_stack([logit(ps), logit(px)]))[:, 1]
            thr, _ = best_threshold(y[va], s[va])
            put(t, f'{name} + XGB_full (stacking)', te, s, (s > thr).astype(np.int8))
    print(f'Fold {k}: {(time.time() - t_all) / 60:.1f} min')
pd.DataFrame(info).to_csv(os.path.join(RESULTS_DIR, 'sequence_models_training.csv'), index=False)

Fold 0: 11.7 min


Fold 1: 26.1 min


Fold 2: 42.9 min


Fold 3: 54.5 min


Fold 4: 71.4 min


## Results

Differences are relative to the GRU (for single models) and to the GRU + XGB_full stacking (for ensembles); 95% CI: patient-level bootstrap.

In [3]:
rows = []
for t in SEQ_TARGETS:
    y = win[f'y_{t}'].values.astype(float)
    labels = list(store[t])
    models = [(store[t][n][0], store[t][n][1].astype(float)) for n in labels]
    B = boot_eval(y, pid, models, n_boot=1000)
    for j, lab in enumerate(labels):
        ref = 'GRU + XGB_full (stacking)' if 'stacking' in lab else 'GRU'
        jr = labels.index(ref)
        d = B[:, j, 1] - B[:, jr, 1]
        rows.append({'target': t, 'model': lab, 'AUROC': roc_auc_score(y, models[j][0]),
                     'AUPRC': average_precision_score(y, models[j][0]), 'AUPRC 95% CI': ci(B[:, j, 1]),
                     'macro_F1': f1_score(y, models[j][1], average='macro'),
                     'ΔAUPRC vs ' + ref: '' if lab == ref else f'{np.mean(d):+.4f} {ci(d)}'})
seq = pd.DataFrame(rows)
seq.to_csv(os.path.join(RESULTS_DIR, 'sequence_models.csv'), index=False)
pd.set_option('display.max_colwidth', 60)
display(seq.round(4))
display(pd.DataFrame(info).groupby(['target', 'model'])[['epochs', 'parameters', 'seconds']].mean().round(1))

,target,model,AUROC,AUPRC,AUPRC 95% CI,macro_F1,ΔAUPRC vs GRU,ΔAUPRC vs GRU + XGB_full (stacking)
0,A,GRU,0.9164,0.4748,[0.446; 0.501],0.7304,,NaN
1,A,GRU + XGB_full (stacking),0.9345,0.5446,[0.517; 0.570],0.7537,NaN,
2,A,TimeAwareGRU,0.9170,0.4684,[0.438; 0.495],0.7304,-0.0063 [-0.010; -0.003],NaN
3,A,TimeAwareGRU + XGB_full (stacking),0.9346,0.5444,[0.516; 0.570],0.7536,NaN,-0.0002 [-0.001; 0.000]
4,A,TCN,0.9163,0.4690,[0.440; 0.496],0.7311,-0.0058 [-0.010; -0.001],NaN
5,A,TCN + XGB_full (stacking),0.9346,0.5439,[0.516; 0.569],0.7535,NaN,-0.0007 [-0.001; -0.000]
6,A,Transformer,0.9184,0.4743,[0.445; 0.501],0.7325,-0.0006 [-0.006; 0.005],NaN
7,A,Transformer + XGB_full (stacking),0.9347,0.5444,[0.516; 0.570],0.7536,NaN,-0.0003 [-0.001; 0.000]
8,B,GRU,0.9571,0.8174,[0.807; 0.828],0.8513,,NaN
9,B,GRU + XGB_full (stacking),0.9656,0.8537,[0.845; 0.862],0.8640,NaN,


epochs  parameters  seconds
target model                                    
A      TCN             10.4     49921.0     73.2
       TimeAwareGRU    24.0     18050.0     66.0
       Transformer     10.8     68673.0     83.9
B      TCN             10.6     49921.0    215.1
       TimeAwareGRU    21.8     18050.0    167.9
       Transformer     11.2     68673.0    244.6